In [ ]:
from typing import List, Dict, Optional

# --- Term representation ---
class Term:
    def __init__(self, name: str, args: Optional[List['Term']] = None):
        self.name = name
        self.args = args or []

    def is_variable(self) -> bool:
        return self.name[0].isupper() and not self.args

    def is_constant(self) -> bool:
        return self.name[0].islower() and not self.args

    def is_compound(self) -> bool:
        return bool(self.args)

    def __repr__(self):
        if self.args:
            return f"{self.name}({', '.join(map(str, self.args))})"
        return self.name

    # --- Enable equality for retract and dict/set usage ---
    def __eq__(self, other):
        if not isinstance(other, Term):
            return False
        return self.name == other.name and self.args == other.args

    def __hash__(self):
        return hash((self.name, tuple(self.args)))


# --- Parsing ---
def parse_term(s: str) -> Term:
    s = s.strip()
    if "(" not in s:
        return Term(s)
    name, rest = s.split("(", 1)
    args_str = rest[:-1]  # remove trailing ")"
    args = [parse_term(a.strip()) for a in split_args(args_str)]
    return Term(name.strip(), args)

def split_args(s: str) -> List[str]:
    args, depth, start = [], 0, 0
    for i, ch in enumerate(s):
        if ch == "(":
            depth += 1
        elif ch == ")":
            depth -= 1
            if depth < 0:
                raise ValueError("Mismatched parentheses")
        elif ch == "," and depth == 0:
            args.append(s[start:i])
            start = i + 1
    if start < len(s):
        args.append(s[start:])
    return [a.strip() for a in args if a.strip()]

def parse_goals(s: str) -> List[Term]:
    s = s.strip().rstrip(".")
    if not s:
        return []
    return [parse_term(g) for g in split_args(s)]


# --- Unification ---
def unify(x: Term, y: Term, subs: Dict[str, Term]) -> Optional[Dict[str, Term]]:
    if subs is None:
        return None
    if x.is_variable():
        return unify_var(x, y, subs)
    elif y.is_variable():
        return unify_var(y, x, subs)
    elif x.is_constant() and y.is_constant():
        return subs if x.name == y.name else None
    elif x.is_compound() and y.is_compound() and x.name == y.name and len(x.args) == len(y.args):
        for a, b in zip(x.args, y.args):
            subs = unify(apply_subs(a, subs), apply_subs(b, subs), subs)
            if subs is None:
                return None
        return subs
    return None

def unify_var(var: Term, x: Term, subs: Dict[str, Term]) -> Optional[Dict[str, Term]]:
    if var.name in subs:
        return unify(subs[var.name], x, subs)
    elif x.is_variable() and x.name in subs:
        return unify(var, subs[x.name], subs)
    elif occurs_check(var, x, subs):
        return None
    else:
        new_subs = subs.copy()
        new_subs[var.name] = x
        return new_subs

def occurs_check(var: Term, x: Term, subs: Dict[str, Term]) -> bool:
    if var == x:
        return True
    if x.is_variable() and x.name in subs:
        return occurs_check(var, subs[x.name], subs)
    if x.is_compound():
        return any(occurs_check(var, arg, subs) for arg in x.args)
    return False

def apply_subs(term: Term, subs: Dict[str, Term]) -> Term:
    if term.is_variable() and term.name in subs:
        return apply_subs(subs[term.name], subs)
    elif term.is_compound():
        return Term(term.name, [apply_subs(arg, subs) for arg in term.args])
    else:
        return term


# --- Knowledge Base ---
class KnowledgeBase:
    def __init__(self):
        self.clauses = []

    def assertz(self, clause: str):
        clause = clause.strip().rstrip(".")
        if ":-" in clause:
            head_str, body_str = clause.split(":-")
            head = parse_term(head_str.strip())
            body = [parse_term(b.strip()) for b in split_args(body_str)]
        else:
            head = parse_term(clause)
            body = []
        self.clauses.append((head, body))

    def retract(self, fact: str):
        fact = fact.strip().rstrip(".")
        head = parse_term(fact)
        self.clauses = [(h, b) for (h, b) in self.clauses if not (h == head and not b)]

    def query(self, q: str, verbose: bool = False, abduction: bool = False) -> List[Dict[str, Term]]:
        goals = parse_goals(q)
        results = []
        proof_paths = []

        if verbose:
            print(f"Query: {q}")

        self._resolve(goals, {}, results, depth=0, verbose=verbose, abduction=abduction, path=[], proof_paths=proof_paths)

        if abduction and proof_paths:
            print("\nExplanation (proof paths) for the query:")
            for i, path in enumerate(proof_paths, 1):
                print(f"Solution {i}:")
                for step in path:
                    print("  ", step)

        query_vars = sorted({v.name for g in goals for v in collect_vars(g)})
        return [{v: apply_subs(Term(v), s) for v in query_vars if v in s} for s in results]

    def _resolve(self, goals: List[Term], subs: Dict[str, Term], results: List[Dict[str, Term]],
                 depth: int, verbose: bool, abduction: bool, path: List[str], proof_paths: List[List[str]]):
        indent = "  " * depth
        if not goals:
            if verbose:
                print(f"{indent}Success with subs: {subs}")
            if abduction:
                proof_paths.append(path.copy())
            results.append(subs)
            return

        first, rest = goals[0], goals[1:]
        if verbose:
            print(f"{indent}Resolving: {apply_subs(first, subs)} with goals {rest}")

        for head, body in self.clauses:
            renamed_head, renamed_body = rename_vars(head, body)
            new_subs = unify(apply_subs(first, subs), renamed_head, subs)
            if new_subs is not None:
                if verbose:
                    print(f"{indent}Matched with clause: {renamed_head} :- {renamed_body}")

                # record the step with applied substitutions for abduction
                step_str = f"{apply_subs(renamed_head, new_subs)} :- {[apply_subs(b, new_subs) for b in renamed_body]}"
                new_path = path + [step_str]

                self._resolve(renamed_body + rest, new_subs, results, depth + 1,
                              verbose, abduction, new_path, proof_paths)
            else:
                if verbose:
                    print(f"{indent}Clause {renamed_head} :- {renamed_body} did not match")


# --- Utilities ---
def rename_vars(head: Term, body: List[Term]):
    counter = rename_vars.counter
    rename_vars.counter += 1
    mapping = {}

    def rename(term: Term):
        if term.is_variable():
            if term.name not in mapping:
                mapping[term.name] = Term(f"{term.name}_{counter}")
            return mapping[term.name]
        elif term.is_compound():
            return Term(term.name, [rename(arg) for arg in term.args])
        return term

    return rename(head), [rename(b) for b in body]
rename_vars.counter = 0

def collect_vars(term: Term):
    if term.is_variable():
        return [term]
    elif term.is_compound():
        return sum((collect_vars(arg) for arg in term.args), [])
    return []
